# CS781 Phase 7 — MORE PREDICTION FILES in about 2 minutes (CPU only, nothing is trained, no GPU needed)

The fast notebook wrote one file (the ensemble of the six runs, dev micro-F1@7 0.3635) because the other two stages did not clear their rules. This notebook builds **several different prediction files** from the same stored scores of Part 1 (no model, no tokenisation): ensembles combined in different ways, the ensemble with pool-aware re-ranking, and, in a separate lexical-hybrid track, combinations with BM25.
Each file: `prediction_<name>.json` and `submission_<name>.zip` (`prediction.json` at the root of the zip), 7 unique candidates per test query from the TEST pool, validated. `prediction.json` / `prediction.zip` = the best Phase 1-only variant on dev. A table at the end shows every variant's dev score, a paired interval against the six-run ensemble and how many test candidates differ.

| variant | what | Phase 1 only? |
|---|---|---|
| ens6_mean | mean of the six runs' standardised scores (your current file) | yes |
| ens2_top2 | the best two runs only | yes |
| ens6_rrf | reciprocal-rank fusion of the six runs | yes |
| ens6_pool_sink, ens6_pool_rrank | the ensemble + pool-aware re-ranking | yes |
| HYBRID_ens6_lex | ensemble + lexical L (BM25 on words + BM25 on word bigrams), one weight chosen on dev | **no, lexical** |
| HYBRID_pool_of_fused | pool-aware re-ranking of that mix | **no, lexical** |
| HYBRID_ens6_plus_pooled_lex | ensemble + pool-aware lexical L | **no, lexical** |

Why the hybrids: in your last run pool-aware re-ranking lifted plain BM25 from 0.2123 to 0.4069 on dev (interval +0.156 to +0.233), above every neural system, but did nothing for the trained models. Whether the combination beats the ensemble is not known until the dev numbers are printed. Submit a hybrid only if the TAs allow lexical scores in the final ranking.

## Inputs
The notebook output of Part 1 (Add Input > Notebook Output Files; the folder `pcr/`), Secrets `HF_TOKEN` (the data files), Internet ON. **GPU off is fine.**

## Honest notes
Pool-aware re-ranking uses only the scores of the split's own unlabeled queries (transductive; mention it in the report). The hybrid weights are chosen on dev (one parameter each; 118 dev queries are few: read the interval, not just the mean). **Tested** on CPU with synthetic data and a stand-in for Part 1's output; not on the real data. Timing is an estimate (about 1.5-2.5 minutes).


In [ ]:
# ================= CELL 0 — CONFIG  [RUNS NOW]. Edit this cell only. =================
HF_DATASET = "Exploration-Lab/CS-781-Capstone"                      # gated Hugging Face dataset (accept its conditions first)
DATA_SUBDIR = "Phase_7_PCR"                                         # the Phase 7 folder inside the dataset repo (Train/, Dev/, Test/)
DATA_SOURCE = "hub_files"                                           # "hub_files" = download the six Phase 7 json files from the dataset repo; "files" = read them from LOCAL_DATA_DIR
LOCAL_DATA_DIR = None                                               # e.g. "/kaggle/input/my-phase7-files" (also the fallback if the download fails)
CKPT_PATHS = {"gpt2_new": None, "modernbert": None, "t5": None, "bilstm": None}   # Phase 1 final_model.pt paths; None = found automatically under /kaggle/input (only ModernBERT is used here)
TOKENIZER_PATH = None                                               # legal_bpe_32k_mask.json; None = found automatically
LSI_WORK = "/kaggle/working/pcr_multi"                                    # results, ledger, rankings, checkpoints (the name is shared with the reused setup cell)
TOP_K = 7                                                           # the evaluator scores exactly 7 candidates per query
TIME_PROFILE = "normal"                                             # "normal" (about 4-5 h) or "fast" (about 2.5 h: shorter training caps, fewer chunks); the notebook also has a hard time budget (settings cell)
RULE_MODE = "strict"                                                # "strict": prediction.json only from Phase 1 models (+ their ensembles); the best Phase 1 + BM25 hybrid is ALSO written to prediction_hybrid.json for you to decide. "hybrid_ok": hybrids may become prediction.json. Plain BM25 is NEVER a submission.
RERUN = False                                                       # True = train runs again even if an attached earlier output already holds them
SELFTEST = False
NB_T0 = __import__("time").time()                                   # start of the time budget


In [ ]:
# ================= CELL 1 — setup (Phase 7: Prior Case Retrieval, Part 2) =================
import os, sys, re, json, math, time, random, glob, gc, copy, hashlib, pickle, getpass, zipfile, shutil
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda"
WORK = Path(LSI_WORK); (WORK / "runs").mkdir(parents=True, exist_ok=True); (WORK / "cache").mkdir(exist_ok=True); SUBDIR = WORK / "submissions"; SUBDIR.mkdir(exist_ok=True)
LEDGER = WORK / "ledger.csv"
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu (training cells need a GPU)", "| work dir:", WORK)


In [ ]:
# ===== CREDENTIALS CHECK  [RUNS NOW; it never asks you to type anything]: reads HF_TOKEN (and optionally KAGGLE_USERNAME / KAGGLE_KEY) from Kaggle Secrets, verifies them, and stops early with the exact fix =====
# Needs Internet ON. Add the Secrets under Add-ons > Secrets with 'Attached' ON for THIS notebook (a committed run only sees attached secrets). The Hugging Face token is not needed when DATA_SOURCE = "files".
import os, subprocess, shutil
REQUIRE_HF, REQUIRE_KAGGLE = (DATA_SOURCE in ("hf", "hub_files")), False        # the Kaggle key is only needed for the optional backup upload (set REQUIRE_KAGGLE = True to insist)
def get_secret(name):
    v = os.environ.get(name)
    if not v:
        try:
            from kaggle_secrets import UserSecretsClient; v = UserSecretsClient().get_secret(name)
        except Exception: v = None
    return (v or "").strip().strip("'\"") or None
problems, report = [], []
hf = get_secret("HF_TOKEN"); HF_OK = False
if DATA_SOURCE not in ("hf", "hub_files"): report.append("Hugging Face token : not needed (DATA_SOURCE = 'files')"); HF_OK = True
elif not hf: report.append("Hugging Face token : MISSING"); problems.append("HF_TOKEN is missing: add a Secret named HF_TOKEN (a token of the account that was approved for the dataset), or set DATA_SOURCE = 'files'")
else:
    try:
        from huggingface_hub import HfApi; api = HfApi(); me = api.whoami(token=hf); api.auth_check(HF_DATASET, repo_type="dataset", token=hf); HF_OK = True; os.environ["HF_TOKEN"] = hf
        report.append(f"Hugging Face token : OK (user '{me.get('name')}', access to {HF_DATASET} confirmed, {len(hf)} characters)")
    except Exception as e:
        code = getattr(getattr(e, "response", None), "status_code", None)
        why = "Hugging Face does not recognise it (wrong, expired or revoked: create a new classic 'Read' token and update the Secret)" if code == 401 else "valid token, but this account has no approved access to the gated dataset" if code == 403 else f"{type(e).__name__}: {str(e)[:100]}"
        report.append(f"Hugging Face token : REFUSED (HTTP {code}): {why}"); problems.append(f"HF_TOKEN does not work: {why}")
ku, kk = get_secret("KAGGLE_USERNAME"), get_secret("KAGGLE_KEY"); KAGGLE_OK = False
if not (ku and kk): report.append("Kaggle credentials  : not provided (the backup upload will be a dry run)"); (problems.append("KAGGLE_USERNAME / KAGGLE_KEY are missing") if REQUIRE_KAGGLE else None)
elif not shutil.which("kaggle"): report.append("Kaggle credentials  : cannot be verified (the kaggle command is not installed)"); (problems.append("the kaggle command is not installed") if REQUIRE_KAGGLE else None)
else:
    try:
        r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True, timeout=120, env={**os.environ, "KAGGLE_USERNAME": ku, "KAGGLE_KEY": kk})
        if r.returncode == 0: KAGGLE_OK = True; os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = ku, kk; report.append(f"Kaggle credentials  : OK (user '{ku}')")
        else: report.append(f"Kaggle credentials  : REFUSED ({(r.stdout + r.stderr).strip()[:100]})"); (problems.append("the Kaggle username/key were refused (use the LEGACY API key; Internet ON?)") if REQUIRE_KAGGLE else None)
    except Exception as e: report.append(f"Kaggle credentials  : not verified ({type(e).__name__})")
print("\n".join(report))
blocking = [p for p in problems if (p.startswith("HF_TOKEN") and REQUIRE_HF) or (not p.startswith("HF_TOKEN") and REQUIRE_KAGGLE)]
if blocking: raise RuntimeError("STOPPING EARLY (before any long step):\n  - " + "\n  - ".join(blocking) + "\nFix: Add-ons > Secrets > add the missing secret(s), switch 'Attached' ON for this notebook, and run again.")
print("\nCredentials ready.")


In [ ]:
# ================= CELL 4 — LOAD AND INSPECT THE PHASE 7 DATA  [RUNS NOW; CPU] =================
# Six files, each split with ITS OWN candidate pool (train.json + train_candidates.json, dev.json + dev_candidates.json, test.json + test_candidates.json). The schema is DETECTED, not assumed: candidate files may be a dict id -> text or a list of records;
# text may be a string or a list of sentences; labels may be strings or numbers (matched to the pool's ids, so zero padding is preserved). Test labels, if a test file carries any, are IGNORED.
FILES = {"train": ("train.json", "train_candidates.json"), "dev": ("dev.json", "dev_candidates.json"), "test": ("test.json", "test_candidates.json")}
def _find(name, roots):
    hits = [p for r in roots for p in glob.glob(f"{r}/**/*", recursive=True) if os.path.basename(p).lower() == name.lower() and "phase" in p.lower() and "7" in p]
    return min(hits, key=len) if hits else None
roots = [r for r in ([LOCAL_DATA_DIR] if LOCAL_DATA_DIR else []) + ["/kaggle/input", str(WORK / "hf")] if r]
if DATA_SOURCE == "hub_files" and not all(_find(f, roots) for pair in FILES.values() for f in pair):
    from huggingface_hub import snapshot_download
    snapshot_download(HF_DATASET, repo_type="dataset", allow_patterns=[f"{DATA_SUBDIR}/*", f"{DATA_SUBDIR}/**"], token=os.environ.get("HF_TOKEN") or get_secret("HF_TOKEN"), local_dir=str(WORK / "hf")); roots.append(str(WORK / "hf"))
PATHS = {s: tuple(_find(f, roots) for f in pair) for s, pair in FILES.items()}; missing = [f for s, pair in FILES.items() for f, p in zip(pair, PATHS[s]) if not p]
assert not missing, f"Phase 7 files not found: {missing} (searched {roots}); attach them or check DATA_SUBDIR / LOCAL_DATA_DIR"
def _load(p):
    t = open(p, encoding="utf-8").read().strip()
    try: return json.loads(t)
    except json.JSONDecodeError: return [json.loads(l) for l in t.splitlines() if l.strip()]
def _text(v): return v if isinstance(v, str) else " ".join(_text(x) for x in v) if isinstance(v, (list, tuple)) else " ".join(_text(x) for x in v.values()) if isinstance(v, dict) else str(v)
ID_KEYS = ("id", "query_id", "case_id", "candidate_id", "doc_id", "qid", "cid"); TEXT_KEYS = ("text", "sentences", "document", "content", "body", "judgment", "doc")
LAB_KEYS = ("relevant_candidates", "relevant_candidate_ids", "relevant", "relevant_ids", "labels", "label", "positives", "gold", "citations", "cited", "relevant_cases")
def _records(obj):
    if isinstance(obj, dict):
        for k in ("data", "queries", "candidates", "records", "items"):
            if isinstance(obj.get(k), list): return obj[k], None
        vals = list(obj.values())[:20]
        if vals and all(isinstance(v, (str, list)) for v in vals): return [{"id": k, "text": v} for k, v in obj.items()], "dict id -> text"
        if vals and all(isinstance(v, dict) for v in vals): return [{"id": k, **v} for k, v in obj.items()], "dict id -> record"
        raise ValueError("unrecognised json object layout")
    return obj, None
def parse(obj, with_labels):
    recs, how = _records(obj); r0 = recs[0]; ik = next(k for k in ID_KEYS if k in r0); tk = next(k for k in TEXT_KEYS if k in r0); lk = next((k for k in LAB_KEYS if k in r0), None) if with_labels else None
    return [r[ik] for r in recs], [_text(r[tk]) for r in recs], ([r.get(lk) or [] for r in recs] if lk else None), dict(id=ik, text=tk, labels=lk, layout=how or "list of records")
DATA, SCHEMA = {}, {}
for s, (qp, cp) in PATHS.items():
    q_raw, q_text, q_lab, qs = parse(_load(qp), with_labels=(s != "test")); c_raw, c_text, _, cs = parse(_load(cp), with_labels=False)
    if s == "test" and any(k in _records(_load(qp))[0][0] for k in LAB_KEYS): print("[test] NOTE: test.json carries a relevance field. Test labels are private, so it is IGNORED and never read.")
    c_ids = [str(x) for x in c_raw]; by_int = {}
    for c in c_ids:
        if c.isdigit(): by_int.setdefault(int(c), c)
    def norm(lbl):
        out = []
        for x in (lbl if isinstance(lbl, (list, tuple, set)) else [lbl]):
            x = str(x)
            if x in pos_c: out.append(x)
            elif x.isdigit() and int(x) in by_int: out.append(by_int[int(x)])
            else: out.append(None)
        return out
    pos_c = {c: i for i, c in enumerate(c_ids)}; gold = None; lost = 0
    if q_lab is not None:
        gold = []
        for l in q_lab: n_ = norm(l); lost += sum(x is None for x in n_); gold.append({pos_c[x] for x in n_ if x is not None})
    DATA[s] = dict(q_ids=[str(x) for x in q_raw], q_raw=q_raw, q_text=q_text, gold=gold, c_ids=c_ids, c_raw=c_raw, c_text=c_text, c_pos=pos_c, lost_labels=lost); SCHEMA[s] = dict(queries=qs, candidates=cs)
print("SCHEMA (detected):", json.dumps(SCHEMA, indent=None)[:600])
rows = []
for s, D in DATA.items():
    wl_q, wl_c = [len(t.split()) for t in D["q_text"]], [len(t.split()) for t in D["c_text"]]; self_in = sum(q in D["c_pos"] for q in D["q_ids"])
    g = [len(x) for x in D["gold"]] if D["gold"] else []
    rows.append(dict(split=s, queries=len(D["q_ids"]), candidates=len(D["c_ids"]), query_words_p50=int(np.median(wl_q)), cand_words_p50=int(np.median(wl_c)), cand_words_p90=int(np.percentile(wl_c, 90)), gold_mean=round(float(np.mean(g)), 2) if g else None,
                     gold_p50=float(np.median(g)) if g else None, gold_max=max(g) if g else None, no_gold=sum(x == 0 for x in g) if g else None, labels_not_in_pool=D["lost_labels"], queries_inside_own_pool=self_in, dup_cand_ids=len(D["c_ids"]) - len(set(D["c_ids"]))))
print(pd.DataFrame(rows).to_string(index=False))
pools = {s: set(D["c_ids"]) for s, D in DATA.items()}; print("candidate-pool overlap between splits:", {f"{a}&{b}": len(pools[a] & pools[b]) for a, b in (("train", "dev"), ("train", "test"), ("dev", "test"))}, "| every split is retrieved ONLY from its own pool")
GOLD_MEAN = float(np.mean([len(x) for x in DATA["dev"]["gold"]])) if DATA["dev"]["gold"] else None; print(f"dev: a query has {GOLD_MEAN:.2f} relevant candidates on average; the evaluator compares the top {TOP_K} with the WHOLE gold set (micro-F1@{TOP_K})")


In [ ]:
# ================= CELL 8 — METRICS, RANKING AND THE SUBMISSION FILE  [pure code] =================
# micro-P@7 = sum of hits / (7 x queries); micro-R@7 = sum of hits / sum of gold sizes (the WHOLE gold set, neither truncated nor padded); micro-F1@7 = their harmonic mean.
def rank_top(scores, D, k=TOP_K):
    """scores [n_queries, n_candidates] -> for every query the indices of its top-k candidates (the query itself is never returned if it sits in its own pool)"""
    S = np.array(scores, dtype=np.float32, copy=True)
    for qi, q in enumerate(D["q_ids"]):
        if q in D["c_pos"]: S[qi, D["c_pos"][q]] = -np.inf
    return np.argsort(-S, axis=1)[:, :k]
def micro_at_k(top, gold, k=TOP_K):
    hits = np.array([len(set(t[:k].tolist()) & g) for t, g in zip(top, gold)], dtype=float); gs = np.array([len(g) for g in gold], dtype=float)
    p = hits.sum() / (k * len(gold)); r = hits.sum() / max(1.0, gs.sum()); f = 0.0 if p + r == 0 else 2 * p * r / (p + r); return dict(f1=f, p=p, r=r, hits=hits, gsize=gs)
def boot_diff(m_a, m_b, n=1000, seed=0, k=TOP_K):
    """paired bootstrap over QUERIES of the micro-F1@k difference a - b: (mean, 2.5%, 97.5%)"""
    r = np.random.default_rng(seed); ha, hb, gs = m_a["hits"], m_b["hits"], m_a["gsize"]; nq = len(gs); out = []
    for _ in range(n):
        i = r.integers(0, nq, nq); f = lambda h: (lambda p, rr: 0.0 if p + rr == 0 else 2 * p * rr / (p + rr))(h[i].sum() / (k * nq), h[i].sum() / max(1.0, gs[i].sum())); out.append(f(ha) - f(hb))
    out = np.array(out); return float(out.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))
def write_prediction(path, top, D):
    sub = {D["q_raw"][qi] if isinstance(D["q_raw"][qi], str) else str(D["q_raw"][qi]): [D["c_raw"][ci] if isinstance(D["c_raw"][ci], str) else str(D["c_raw"][ci]) for ci in top[qi][:TOP_K]] for qi in range(len(D["q_ids"]))}
    json.dump(sub, open(path, "w", encoding="utf-8"), ensure_ascii=False); return sub
def validate_prediction(path, D):
    sub = json.load(open(path, encoding="utf-8")); pool = set(map(str, D["c_raw"])); qs = list(map(str, D["q_raw"]))
    assert isinstance(sub, dict), "prediction.json must be a JSON object: query id -> list of candidate ids"
    assert set(sub) == set(qs) and len(sub) == len(qs), f"query ids differ from the test file ({len(set(sub) ^ set(qs))} differ)"
    for q, lst in sub.items():
        assert isinstance(lst, list) and len(lst) == TOP_K, f"{q}: {len(lst)} candidates instead of {TOP_K}"
        assert len(set(lst)) == TOP_K, f"{q}: duplicate candidates"; assert all(c in pool for c in lst), f"{q}: a candidate outside the TEST pool"; assert q not in lst, f"{q}: the query itself was returned"
    return True
def zip_prediction(path, zpath):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(path, arcname="prediction.json")          # the file at the ROOT of the archive, no folder


In [ ]:
# ================= CELL 5 — FUNCTIONS (reused, tested cells of Parts 1-2)  [pure code] =================
# BM25 on words and on word bigrams (hashed), and the pool-aware re-ranking (z-normalise per query, then penalise a candidate that other queries of the same split match better: 'sink' = Sinkhorn normalisation, 'rrank' = reciprocal rank).
import itertools
from sklearn.feature_extraction.text import CountVectorizer
import scipy.sparse as sp
STOP = "the of and to in a is that be by for was as it on with or this not he which his are has an at from any have under were been had shall said such no but if may their they there its upon would other these those than also into made court section".split()
def bm25_matrix(D, k1=1.2, b=0.75, fitted=None):
    if fitted is None:
        cv = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z0-9][a-z0-9]+\b", stop_words=STOP, dtype=np.float32, min_df=1); C = cv.fit_transform(D["c_text"]).tocsr(); Q = cv.transform(D["q_text"]).tocsr(); fitted = (C, Q)
    C, Q = fitted; N = C.shape[0]; dl = np.asarray(C.sum(1)).ravel(); avg = dl.mean() + 1e-9; dfreq = np.asarray((C > 0).sum(0)).ravel(); idf = np.log((N - dfreq + 0.5) / (dfreq + 0.5) + 1.0).astype(np.float32)
    Co = C.tocoo(); w = Co.data * (k1 + 1) / (Co.data + k1 * (1 - b + b * dl[Co.row] / avg)) * idf[Co.col]; W = sp.csr_matrix((w.astype(np.float32), (Co.row, Co.col)), shape=C.shape)
    Qw = Q.copy(); Qw.data = np.log1p(Qw.data); return np.asarray((Qw @ W.T).todense(), dtype=np.float32), fitted

from sklearn.feature_extraction.text import HashingVectorizer
import itertools
def bm25_ng(D, n):
    hv = HashingVectorizer(analyzer="word", ngram_range=(n, n), n_features=2 ** 22, alternate_sign=False, norm=None, lowercase=True, token_pattern=r"(?u)\b[a-z0-9][a-z0-9]+\b", stop_words=STOP, dtype=np.float32)
    return bm25_matrix(D, BM_K1, BM_B, (hv.transform(D["c_text"]).tocsr(), hv.transform(D["q_text"]).tocsr()))[0]

def zrows(S, D):
    S = np.array(S, dtype=np.float32, copy=True)
    for qi, q in enumerate(D["q_ids"]):
        if q in D["c_pos"]: S[qi, D["c_pos"][q]] = np.nan
    Z = (S - np.nanmean(S, 1, keepdims=True)) / (np.nanstd(S, 1, keepdims=True) + 1e-6); return np.nan_to_num(Z, nan=-1e4)
def pool_rerank(S, D, mode, p):
    Z = zrows(S, D)
    if mode == "lse": tau, beta = p; Zt = Z / tau; m = Zt.max(0, keepdims=True); L = tau * (m[0] + np.log(np.exp(Zt - m).sum(0))); return Z - beta * L[None, :]
    if mode == "csls": k, beta = p; top = (-np.sort(-Z, axis=0)[:k]).mean(0); return Z - beta * top[None, :]
    if mode == "rrank":                                                              # RECIPROCAL RANK: how highly this candidate ranks for the query AND how highly this query ranks among all queries for the candidate (a candidate belongs to ~one query)
        lam, _ = p; R = np.argsort(np.argsort(-Z, axis=1), axis=1); C = np.argsort(np.argsort(-Z, axis=0), axis=0); return -(np.log1p(R) + lam * np.log1p(C)) + 1e-3 * Z
    if mode == "sink":
        tau, it = p; K = np.exp(np.clip(Z / tau, -50, 50)); 
        for _ in range(it): K = K / (K.sum(1, keepdims=True) + 1e-12); K = K / (K.sum(0, keepdims=True) + 1e-12)
        return np.log(K + 1e-30)
    raise ValueError(mode)

BM_K1, BM_B = 1.6, 0.9                                   # the BM25 settings chosen on dev in the earlier run (k1 1.6, b 0.9)
POOL_SINK, POOL_RR = ("sink", (0.5, 3)), ("rrank", (1.0, 0))      # 'sink' was the setting chosen on the TRAINING split with BM25 alone (no dev label); 'rrank' was the best dev-tuned one in the earlier run


In [ ]:
# ================= CELL 6 — THE STORED SCORES AND THE PHASE 1 VARIANTS  [RUNS NOW; CPU; seconds] =================
# Reads the dev and test score matrices Part 1 stored for every run (no model, no GPU, no tokenisation) and builds DIFFERENT rule-safe rankings from the same six retrievers:
#   ens6_mean  mean of the per-query standardised scores of all six runs (the file you already have)        ens2_top2  the best two runs only
#   ens6_rrf   reciprocal-rank fusion of the six runs (a different way to combine them)                      ens6_pool_sink / ens6_pool_rrank  the same ensemble + pool-aware re-ranking
T_START = time.time(); NEURAL_TECH = ["full", "partial", "lorma_plus", "ia3", "prompt", "kd"]
SRCS = sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/pcr/ledger.csv", recursive=True)}, key=lambda p: -len(list((p / "runs").glob("*/result.json")))); assert SRCS, "attach the output of Part 1 (Add Input > Notebook Output Files): the folder pcr/ with runs/"
RUNS = {}
for src in SRCS:
    for p in (src / "runs").glob("*/result.json"):
        r = json.load(open(p)); RUNS.setdefault(r["name"], dict(r, root=src))
NEURAL = sorted([n for n, r in RUNS.items() if r.get("technique") in NEURAL_TECH and not r.get("control")], key=lambda n: -RUNS[n]["dev_f1"]); assert len(NEURAL) >= 2, f"fewer than two finished neural runs found: {NEURAL}"
print("restored from", SRCS[0], "| neural runs (best first):", [(n, round(RUNS[n]["dev_f1"], 4)) for n in NEURAL])
def sc(n, s): return np.load(RUNS[n]["root"] / "runs" / n / f"{s}_scores.npy").astype(np.float32)
def z(S): S = S.astype(np.float32); return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)
def f1(S, s="dev"): return micro_at_k(rank_top(S, DATA[s]), DATA[s]["gold"])
for s in ("dev", "test"): assert sc(NEURAL[0], s).shape == (len(DATA[s]["q_ids"]), len(DATA[s]["c_ids"])), f"the stored {s} scores do not match this data: attach the Part 1 output that was produced on the same Phase 7 files"
SC = {}
def add(tag, dev, test, lex, note): SC[tag] = dict(dev=dev, test=test, lex=lex, note=note)
def mean_ens(names, s): return sum(z(sc(n, s)) for n in names) / len(names)
def rrf(names, s, k=60): return sum(1.0 / (k + np.argsort(np.argsort(-zrows(sc(n, s), DATA[s]), axis=1), axis=1) + 1) for n in names)
E6 = {s: mean_ens(NEURAL, s) for s in ("dev", "test")}
add("ens6_mean", E6["dev"], E6["test"], False, f"mean of the standardised scores of all {len(NEURAL)} runs")
add("ens2_top2", mean_ens(NEURAL[:2], "dev"), mean_ens(NEURAL[:2], "test"), False, f"best two runs: {NEURAL[:2]}")
add("ens6_rrf", rrf(NEURAL, "dev"), rrf(NEURAL, "test"), False, "reciprocal-rank fusion (k=60) of all runs")
for tag, par in (("ens6_pool_sink", POOL_SINK), ("ens6_pool_rrank", POOL_RR)): add(tag, pool_rerank(E6["dev"], DATA["dev"], *par), pool_rerank(E6["test"], DATA["test"], *par), False, f"ensemble + pool-aware re-ranking {par}")
for tag in SC: print(f"  {tag:18s} dev micro-F1@7 {f1(SC[tag]['dev'])['f1']:.4f}")


In [ ]:
# ================= CELL 7 — HYBRID VARIANTS (lexical BM25 enters the ranking: NOT Phase 1 models only)  [RUNS NOW; CPU; about 1 min] =================
# In the earlier run pool-aware re-ranking lifted plain BM25 from 0.21 to 0.41 on dev (interval +0.16 to +0.23), above the neural systems, while it did nothing for the trained models (they had absorbed that structure). So the lexical signal may be worth combining.
# L = z(BM25 on words) + z(BM25 on word bigrams) (equal weights, nothing tuned). Three ways to combine it with the six-run ensemble; ONE weight each, chosen on dev. Submit a HYBRID file only if the TAs allow lexical scores in the final ranking.
L = {s: z(bm25_matrix(DATA[s], BM_K1, BM_B)[0]) + z(bm25_ng(DATA[s], 2)) for s in ("dev", "test")}
print(f"lexical L alone: dev micro-F1@7 {f1(L['dev'])['f1']:.4f} | with pool-aware re-ranking {f1(pool_rerank(L['dev'], DATA['dev'], *POOL_SINK))['f1']:.4f}")
alphas = [round(a, 1) for a in np.arange(0.1, 0.91, 0.1)]; betas = [0.25, 0.5, 1.0, 2.0, 4.0]
mk1 = lambda a, s: (1 - a) * z(E6[s]) + a * z(L[s]); mk2 = lambda a, s: pool_rerank(mk1(a, s), DATA[s], *POOL_SINK); mk3 = lambda b, s: z(E6[s]) + b * z(np.maximum(pool_rerank(L[s], DATA[s], *POOL_SINK), -20.0))
for tag, mk, grid, note in (("HYBRID_ens6_lex", mk1, alphas, "(1-a) x ensemble + a x lexical L, a = {}"), ("HYBRID_pool_of_fused", mk2, alphas, "pool-aware re-ranking of (1-a) x ensemble + a x lexical L, a = {}"), ("HYBRID_ens6_plus_pooled_lex", mk3, betas, "ensemble + b x pool-aware lexical L, b = {}")):
    p = max(grid, key=lambda g: f1(mk(g, "dev"))["f1"]); add(tag, mk(p, "dev"), mk(p, "test"), True, note.format(p)); print(f"  {tag:28s} {note.format(p):62s} dev micro-F1@7 {f1(SC[tag]['dev'])['f1']:.4f}")


In [ ]:
# ================= CELL 8 — WRITE EVERY FILE AND THE COMPARISON  [RUNS NOW; CPU; seconds] =================
# For every variant: prediction_<tag>.json and submission_<tag>.zip (prediction.json at the ROOT of the zip), 7 unique candidates per test query from the TEST pool, validated. prediction.json / prediction.zip = the best Phase 1-only variant on dev.
OUT = Path("/kaggle/working"); base = f1(SC["ens6_mean"]["dev"]); ref_top = rank_top(SC["ens6_mean"]["test"], DATA["test"]); rows = []
for tag, v in SC.items():
    m = f1(v["dev"]); d, lo, hi = boot_diff(m, base) if tag != "ens6_mean" else (0.0, 0.0, 0.0); top = rank_top(v["test"], DATA["test"]); ov = float(np.mean([len(set(a.tolist()) & set(b.tolist())) / TOP_K for a, b in zip(top, ref_top)]))
    path = OUT / f"prediction_{tag}.json"; write_prediction(path, top, DATA["test"]); validate_prediction(path, DATA["test"]); zip_prediction(path, OUT / f"submission_{tag}.zip")
    rows.append(dict(file=f"prediction_{tag}.json", uses_lexical_BM25=v["lex"], dev_f1=round(m["f1"], 4), dev_p=round(m["p"], 4), dev_r=round(m["r"], 4), vs_ens6_mean=f"{d:+.4f} [{lo:+.4f}, {hi:+.4f}]", test_top7_overlap_with_ens6=f"{ov:.0%}", how=v["note"]))
TAB = pd.DataFrame(rows); print("\nALL PREDICTION FILES (dev micro-F1@7; the interval is a paired bootstrap over queries against ens6_mean; the overlap says how different the test predictions are):"); print(TAB.to_string(index=False))
safe_rows = TAB[~TAB.uses_lexical_BM25].sort_values("dev_f1", ascending=False); best_safe = safe_rows.iloc[0]["file"].replace("prediction_", "").replace(".json", "")
shutil.copy2(OUT / f"prediction_{best_safe}.json", OUT / "prediction.json"); zip_prediction(OUT / "prediction.json", OUT / "prediction.zip"); validate_prediction(OUT / "prediction.json", DATA["test"])
hy = TAB[TAB.uses_lexical_BM25].sort_values("dev_f1", ascending=False)
print(f"\nprediction.json = prediction_{best_safe}.json (best Phase 1-only variant on dev). Best hybrid on dev: {hy.iloc[0]['file']} ({hy.iloc[0]['dev_f1']:.4f}, {hy.iloc[0]['vs_ens6_mean']} against ens6_mean): hybrids use lexical BM25: submit one only if the TAs allow it.")
print(f"finished in {time.time() - T_START:.0f}s | files:", sorted(p.name for p in OUT.glob('*') if p.name.startswith(('prediction', 'submission'))))
